# LingkodAI on your own recordings (Colab)

Runs your recordings through the full seven-stage pipeline: put audio files
in a Google Drive folder, run the cells, and read the answers (text and
speech) from another Drive folder. This is `scripts/run_folder.py`; see the
README's "Running your own recordings".

**Before you start**

- Runtime: GPU. **L4 preferred** (the golden check ran on one). The pipeline
  runs `staged`, so one model family is on the GPU at a time.
- A Colab Secret named `HF_TOKEN` (key icon in the left sidebar) with notebook
  access on: a fine-grained read-only token from your own Hugging Face
  account, as a `lingkodai` org member, that can read the three ASR model
  repos, `lingkodai/lingkodai-audio-lid-head` and the
  `lingkodai/lingkodai-eval-assets` dataset repo.
- Roughly 45 GB of model downloads on a fresh runtime. Each conversation then
  loads the models again, so expect a few minutes per conversation.

**Where your files go** (step 3 creates both folders):

- `MyDrive/lingkodai_input/`: your recordings. An audio file is a
  one-question conversation; a subfolder is a multi-turn conversation, its
  files taken as turns in filename order (`01_...`, `02_...`). Any format
  ffmpeg reads (m4a, mp3, wav, ...), one question in Cebuano, Filipino or
  English per file.
- `MyDrive/lingkodai_output/`: `summary.csv` (one row per question) and one
  folder per conversation with every stage's output and the spoken answer.
  Recordings that already have results are skipped, so you can add more and
  run step 7 again.

The spoken answers use a cloned voice from a Philippine Languages Database
research speaker (CC-BY-NC, research only). Keep the output folder private.

Run the cells in order.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU (L4 preferred)"
print(torch.cuda.get_device_name(0), "| arch list:", torch.cuda.get_arch_list())

## 2. Get the code and install

Uses Colab's preinstalled torch and installs the rest from `pyproject.toml`
(a few minutes). `qwen-tts` pins `transformers==4.57.3`; pip may downgrade
Colab's copy, which is expected. Every later step runs as a separate `python`
process, so no runtime restart is needed.

In [ ]:
BRANCH = "main"
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg sox > /dev/null
import os
!test -d /content/LingkodAI || git clone -q --branch {BRANCH} https://github.com/isabelbeab/LingkodAI.git /content/LingkodAI
assert os.path.isdir("/content/LingkodAI/scripts"), "git clone failed; see the output above"
%cd /content/LingkodAI
assert os.path.isfile("scripts/run_folder.py"), f"branch {BRANCH!r} has no scripts/run_folder.py: check BRANCH"
!git log --oneline -1
!pip install -q -e ".[gpu]"
assert _exit_code == 0, "pip install failed; see the output above"
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"

## 3. Token, Google Drive and your folders

Reads `HF_TOKEN` from Colab Secrets (never printed or written to disk),
mounts Google Drive (allow the popup) and creates the input and output
folders. Upload your recordings to `MyDrive/lingkodai_input/` at
drive.google.com, or with the Files panel on the left, before step 7.

In [ ]:
%cd /content/LingkodAI
import os
from pathlib import Path
from google.colab import drive, userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert os.environ["HF_TOKEN"], "The HF_TOKEN secret is empty"
print("HF_TOKEN set")

drive.mount("/content/drive")
INPUT = "/content/drive/MyDrive/lingkodai_input"
OUTPUT = "/content/drive/MyDrive/lingkodai_output"
for d in (INPUT, OUTPUT):
    os.makedirs(d, exist_ok=True)
print("input folder :", INPUT)
print("output folder:", OUTPUT)
print("in the input folder now:", sorted(p.name for p in Path(INPUT).iterdir()) or "nothing yet")

## 4. Patch the ASR tokenizers

The fine-tuned Whisper checkpoints do not load as published under
transformers 4.57.3 (`CHANGELOG.md`, 2026-09-22). This builds verified,
patched local copies and points the pipeline at them.

In [ ]:
%cd /content/LingkodAI
!python scripts/patch_asr_tokenizers.py --out outputs/asr_patched
assert _exit_code == 0, "patch_asr_tokenizers.py failed; see the output above"
for lang in ("ceb", "fil", "eng"):
    path = f"/content/LingkodAI/outputs/asr_patched/{lang}"
    assert os.path.isdir(path), f"missing {path}: the patch step failed"
    os.environ[f"ASR_{lang.upper()}"] = path
    print(f"ASR_{lang.upper()}={path}")

## 5. Fetch the reference voices

Downloads the private eval assets, including the three TTS reference voices,
and checks every file against its SHA-256.

In [ ]:
%cd /content/LingkodAI
ASSETS = "/content/LingkodAI/outputs/eval_assets"
!python scripts/fetch_eval_assets.py --out {ASSETS}
assert _exit_code == 0, "fetch_eval_assets.py failed; see the output above"
os.environ["TTS_REF_DIR"] = f"{ASSETS}/tts_ref"
print("TTS_REF_DIR =", os.environ["TTS_REF_DIR"])

## 6. Optional: no recordings yet?

If the input folder is empty, this copies one recording from the golden
conversation into it as `sample_<name>`, so you can try the notebook first.
Skip it if you uploaded your own.

In [ ]:
%cd /content/LingkodAI
import shutil
if any(Path(INPUT).iterdir()):
    print("The input folder already has files; nothing copied.")
else:
    clips = sorted(p for p in Path(ASSETS, "golden", "audio").rglob("*") if p.is_file())
    assert clips, "no golden clips found: did step 5 succeed?"
    sample = Path(INPUT) / f"sample_{clips[0].name}"
    shutil.copy(clips[0], sample)
    print("copied", clips[0].name, "->", sample)

## 7. Run

Every recording without results yet goes through all seven stages. The log
shows each stage and is also saved as `run_folder.log` in the output folder.
Rerun this cell after adding recordings: finished ones are skipped.

In [ ]:
%cd /content/LingkodAI
import shlex
cmd = f"python scripts/run_folder.py --input {shlex.quote(INPUT)} --output {shlex.quote(OUTPUT)}"
print(cmd)
!{cmd} 2>&1 | tee {shlex.quote(OUTPUT + "/run_folder.log")}

## 8. Read and listen

The summary table, then each answer with its audio. A row with no audio fell
back to text only (its status says why) or failed (its error says why).

In [ ]:
%cd /content/LingkodAI
import pandas as pd
from IPython.display import Audio, Markdown, display

summary = pd.read_csv(f"{OUTPUT}/summary.csv", encoding="utf-8-sig").fillna("")
with pd.option_context("display.max_colwidth", 80):
    display(summary[["conversation", "turn", "status", "final_lang", "transcript", "native_answer"]])
for _, row in summary.iterrows():
    display(Markdown(f"### {row['conversation']}, turn {row['turn']} ({row['final_lang']})\n\n"
                     f"**Heard:** {row['transcript']}\n\n**Answer:** {row['native_answer']}"))
    if row["audio_file"]:
        display(Audio(f"{OUTPUT}/{row['audio_file']}"))
    else:
        display(Markdown(f"*No audio: {row['status']} {row['error']}*"))